# NYC 311 — Data Cleaning

This notebook cleans and validates the raw NYC 311 service request
snapshot before feature engineering and machine learning.

Input:
- data/raw/nyc311_2025_sample.csv

Output:
- data/processed/nyc311_2025_cleaned.csv

In [3]:
import pandas as pd
from pathlib import Path

In [4]:
PROJECT_ROOT = Path.cwd().parent

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "nyc311_2025_sample.csv"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "nyc311_2025_cleaned.csv"

In [5]:
print(RAW_PATH)
print(RAW_PATH.exists())

/Users/mahimjohn/Desktop/NYC311-Triage-capstone-project/data/raw/nyc311_2025_sample.csv
True


In [6]:
raw = pd.read_csv(RAW_PATH)

print("Rows:", raw.shape[0])
print("Columns:", raw.shape[1])

Rows: 100000
Columns: 13


In [7]:
required_columns = [
    "unique_key",
    "created_date",
    "agency",
    "agency_name",
    "complaint_type",
    "descriptor",
    "location_type",
    "incident_zip",
    "borough",
    "city",
    "latitude",
    "longitude",
    "closed_date"
]

missing_columns = [
    column for column in required_columns
    if column not in raw.columns
]

unexpected_columns = [
    column for column in raw.columns
    if column not in required_columns
]

print("Missing required columns:", missing_columns)
print("Unexpected columns:", unexpected_columns)

Missing required columns: []
Unexpected columns: []


In [8]:
df = raw.copy()

In [9]:
df["created_date"] = pd.to_datetime(df["created_date"])
df["closed_date"] = pd.to_datetime(df["closed_date"])

In [10]:
print(df[["created_date", "closed_date"]].dtypes)

created_date    datetime64[ns]
closed_date     datetime64[ns]
dtype: object


In [11]:
duplicate_keys = df["unique_key"].duplicated().sum()

print("Duplicate unique keys:", duplicate_keys)

Duplicate unique keys: 0


In [12]:
categorical_columns = [
    "agency",
    "agency_name",
    "complaint_type",
    "descriptor",
    "location_type",
    "borough",
    "city"
]

for column in categorical_columns:
    df[column] = df[column].fillna("Unknown")

In [13]:
df[categorical_columns].isna().sum()

agency            0
agency_name       0
complaint_type    0
descriptor        0
location_type     0
borough           0
city              0
dtype: int64

In [14]:
df["resolution_days"] = (
    df["closed_date"] - df["created_date"]
).dt.total_seconds() / (24 * 60 * 60)

In [15]:
df["resolution_days"].describe()

count    100000.000000
mean          8.386056
std          32.876383
min          -2.243750
25%           0.078238
50%           0.509508
75%           1.990162
max         603.805347
Name: resolution_days, dtype: float64

In [16]:
invalid_duration = df["resolution_days"] < 0

print("Invalid resolution records:", invalid_duration.sum())

Invalid resolution records: 2


In [17]:
df = df[~invalid_duration].copy()

In [18]:
print("Negative resolution times remaining:",
      (df["resolution_days"] < 0).sum())

Negative resolution times remaining: 0


In [19]:
df["delay_flag"] = (df["resolution_days"] > 7).astype(int)

In [20]:
print(df["delay_flag"].value_counts())
print()
print(df["delay_flag"].value_counts(normalize=True) * 100)

delay_flag
0    87947
1    12051
Name: count, dtype: int64

delay_flag
0    87.948759
1    12.051241
Name: proportion, dtype: float64


In [21]:
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate unique keys:",
      df["unique_key"].duplicated().sum())

print("\nNegative resolution times:",
      (df["resolution_days"] < 0).sum())

Rows: 99998
Columns: 15

Missing values:
unique_key           0
created_date         0
agency               0
agency_name          0
complaint_type       0
descriptor           0
location_type        0
incident_zip       473
borough              0
city                 0
latitude           794
longitude          794
closed_date          0
resolution_days      0
delay_flag           0
dtype: int64

Duplicate unique keys: 0

Negative resolution times: 0


In [ ]:
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(PROCESSED_PATH, index=False)

print("Cleaned dataset saved to:")
print(PROCESSED_PATH)

Cleaned dataset saved to:
/Users/mahimjohn/Desktop/NYC311-Triage-capstone-project/data/processed/nyc311_2025_cleaned.csv
